In [ ]:
# main.py
# Warehouse Management System
#
# Requires:
#   pip install mysql-connector-python
#
# IMPORTANT: run schema_update.sql against your database once before
# running this app -- it adds a few columns/tables the new workflows need
# (bin location, dispatch, damagedStock, reconciliation, transfer status).
#
# Workflows covered:
#   Receiving        -> Stock tab, "Receive Stock" (Add)
#   Put-away         -> Stock tab, "Put-away (Set Bin)"
#   Bin Search       -> Stock tab, Search box (matches product/warehouse/bin)
#   Transfer         -> Transfers tab (Initiate) + Transfer Details (line items)
#     Init/Confirm      + "Confirm Selected" button (moves the real stock)
#   Dispatch         -> Dispatch tab
#   Damage Entry     -> Damage Entry tab
#   Reconciliation   -> Reconciliation tab
#   Reporting        -> Report tab (several report types)
#
# Every list tab supports: Add, Edit, Delete, View (Refresh), Search.
# Dispatch / Damage / Reconciliation are audit logs tied to real stock
# movements, so they support Add/Delete (delete reverses the stock change)
# but not Edit -- editing them after the fact would silently break the
# stock numbers they already affected.

import os
from datetime import date, datetime

import tkinter as tk
from tkinter import ttk, messagebox, simpledialog
import mysql.connector

# ---------------------------------------------------------------------------
# DB CONFIG
# ---------------------------------------------------------------------------
DB = {
    "host": "localhost",
    "user": "root",
    "password":("the password"),
    "database": "warehouseDB",
}


def conn():
    return mysql.connector.connect(**DB)


def run(sql, params=(), fetch=False):
    """Simple one-shot query, auto-commit. Used for reads and simple writes."""
    c = conn()
    cur = c.cursor()
    try:
        cur.execute(sql, params)
        data = cur.fetchall() if fetch else None
        c.commit()
        return data
    finally:
        cur.close()
        c.close()


def run_dict(sql, params=()):
    """Read query returning list of dict rows (used to prefill Edit forms)."""
    c = conn()
    cur = c.cursor(dictionary=True)
    try:
        cur.execute(sql, params)
        return cur.fetchall()
    finally:
        cur.close()
        c.close()


def with_transaction(func):
    """Runs func(cursor) inside a single transaction; commits on success,
    rolls back on any exception, and re-raises the exception."""
    c = conn()
    c.autocommit = False
    cur = c.cursor()
    try:
        result = func(cur)
        c.commit()
        return result
    except Exception:
        c.rollback()
        raise
    finally:
        cur.close()
        c.close()


def next_id(table, pk, cur=None):
    sql = f"SELECT COALESCE(MAX({pk}), 0) + 1 FROM {table}"
    if cur is not None:
        cur.execute(sql)
        return cur.fetchone()[0]
    return run(sql, fetch=True)[0][0]


def friendly_db_error(e):
    """Turns common MySQL error codes into plain-English messages."""
    code = getattr(e, "errno", None)
    if code == 1451:
        return "Can't delete this -- other records still reference it. Delete those first."
    if code == 1452:
        return "Can't save this -- it refers to a record that doesn't exist."
    if code == 1062:
        return "That ID already exists. Try again."
    if code == 2003:
        return "Can't reach the database server. Is MySQL running?"
    if code == 1045:
        return "Database login failed -- check the username/password in main.py."
    if code == 1049:
        return "Database 'warehouseDB' doesn't exist. Did you run the setup SQL?"
    return str(e)


# ---------------------------------------------------------------------------
# Validation helpers -- each raises ValueError with a user-facing message
# ---------------------------------------------------------------------------
def v_required(value, label):
    value = (value or "").strip()
    if value == "":
        raise ValueError(f"'{label}' is required.")
    return value


def v_int(value, label, minimum=0):
    value = (value or "").strip()
    if value == "":
        raise ValueError(f"'{label}' is required.")
    try:
        n = int(value)
    except ValueError:
        raise ValueError(f"'{label}' must be a whole number.")
    if minimum is not None and n < minimum:
        raise ValueError(f"'{label}' must be at least {minimum}.")
    return n


def v_email(value, label="Email"):
    value = (value or "").strip()
    if value == "":
        raise ValueError(f"'{label}' is required.")
    if "@" not in value or "." not in value.split("@")[-1]:
        raise ValueError(f"'{label}' doesn't look like a valid email address.")
    return value


def v_date(value, label="Date"):
    value = (value or "").strip()
    if value == "":
        raise ValueError(f"'{label}' is required.")
    try:
        datetime.strptime(value, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"'{label}' must be in YYYY-MM-DD format, e.g. 2026-08-20.")
    return value


VALIDATORS = {
    "text": v_required,
    "int": v_int,
    "email": v_email,
    "date": v_date,
}


# ---------------------------------------------------------------------------
# Base-table metadata: warehouse / supplier / product / stock / transfer /
# transferDetails all go through the generic Add/Edit/Delete/Search tab.
# ---------------------------------------------------------------------------
TABLES = {
    "warehouse": {
        "pk": "warehouseID",
        "display_query": "SELECT warehouseID, warehouseName, location FROM warehouse",
        "labels": ["ID", "Warehouse Name", "Location"],
        "add_fields": [
            {"column": "warehouseName", "label": "Warehouse Name", "kind": "text"},
            {"column": "location", "label": "Location", "kind": "text"},
        ],
    },
    "supplier": {
        "pk": "supplierID",
        "display_query": "SELECT supplierID, supplierName, phone, email FROM supplier",
        "labels": ["ID", "Supplier Name", "Phone", "Email"],
        "add_fields": [
            {"column": "supplierName", "label": "Supplier Name", "kind": "text"},
            {"column": "phone", "label": "Phone", "kind": "int"},
            {"column": "email", "label": "Email", "kind": "email"},
        ],
    },
    "product": {
        "pk": "productID",
        "display_query": """
            SELECT p.productID, p.productName, p.price, p.category, s.supplierName
            FROM product p
            LEFT JOIN supplier s ON p.supplierID = s.supplierID
        """,
        "labels": ["ID", "Product Name", "Price", "Category", "Supplier"],
        "add_fields": [
            {"column": "productName", "label": "Product Name", "kind": "text"},
            {"column": "price", "label": "Price", "kind": "int", "min": 0},
            {"column": "category", "label": "Category", "kind": "text"},
            {"column": "supplierID", "label": "Supplier", "kind": "fk",
             "choices_query": "SELECT supplierID, supplierName FROM supplier ORDER BY supplierName"},
        ],
    },
    "stock": {
        "pk": "stockID",
        "display_query": """
            SELECT s.stockID, w.warehouseName, p.productName, s.quantity,
                   COALESCE(s.binLocation, '')
            FROM stock s
            JOIN warehouse w ON s.warehouseID = w.warehouseID
            JOIN product p ON s.productID = p.productID
        """,
        "labels": ["ID", "Warehouse", "Product", "Quantity", "Bin Location"],
        "add_fields": [
            {"column": "warehouseID", "label": "Warehouse", "kind": "fk",
             "choices_query": "SELECT warehouseID, warehouseName FROM warehouse ORDER BY warehouseName"},
            {"column": "productID", "label": "Product", "kind": "fk",
             "choices_query": "SELECT productID, productName FROM product ORDER BY productName"},
            {"column": "quantity", "label": "Quantity", "kind": "int", "min": 0},
            {"column": "binLocation", "label": "Bin Location", "kind": "text", "required": False},
        ],
        "add_button_label": "Receive Stock",
    },
    "transfer": {
        "pk": "transferID",
        "display_query": """
            SELECT t.transferID, wf.warehouseName, wt.warehouseName, t.transferData, t.status
            FROM transfer t
            JOIN warehouse wf ON t.from_warehouseID = wf.warehouseID
            JOIN warehouse wt ON t.to_warehouseID = wt.warehouseID
        """,
        "labels": ["ID", "From Warehouse", "To Warehouse", "Date (YYYYMMDD)", "Status"],
        "add_fields": [
            {"column": "from_warehouseID", "label": "From Warehouse", "kind": "fk",
             "choices_query": "SELECT warehouseID, warehouseName FROM warehouse ORDER BY warehouseName"},
            {"column": "to_warehouseID", "label": "To Warehouse", "kind": "fk",
             "choices_query": "SELECT warehouseID, warehouseName FROM warehouse ORDER BY warehouseName"},
            {"column": "transferData", "label": "Date (YYYYMMDD)", "kind": "int", "min": 1},
        ],
        "fixed_values_on_insert": {"status": "Pending"},
        "add_button_label": "Initiate Transfer",
    },
    "transferDetails": {
        "pk": "transfer_DetailID",
        "display_query": """
            SELECT td.transfer_DetailID, td.transferID, p.productName, td.quantity
            FROM transferDetails td
            JOIN product p ON td.productID = p.productID
        """,
        "labels": ["ID", "Transfer ID", "Product", "Quantity"],
        "add_fields": [
            {"column": "transferID", "label": "Transfer (pending only)", "kind": "fk",
             "choices_query": "SELECT transferID, CONCAT('Transfer #', transferID) "
                               "FROM transfer WHERE status='Pending' ORDER BY transferID"},
            {"column": "productID", "label": "Product", "kind": "fk",
             "choices_query": "SELECT productID, productName FROM product ORDER BY productName"},
            {"column": "quantity", "label": "Quantity", "kind": "int", "min": 1},
        ],
        "add_button_label": "Add Line Item",
    },
}

TAB_TITLES = {
    "warehouse": "Warehouses",
    "supplier": "Suppliers",
    "product": "Products",
    "stock": "Stock",
    "transfer": "Transfers",
    "transferDetails": "Transfer Details",
}


def stock_choices():
    """(stockID, 'Product @ Warehouse -- Qty: N') for dropdowns."""
    rows = run("""
        SELECT s.stockID, p.productName, w.warehouseName, s.quantity
        FROM stock s
        JOIN product p ON s.productID = p.productID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        ORDER BY p.productName
    """, fetch=True) or []
    return [(sid, f"{pname} @ {wname} (Qty: {qty})") for sid, pname, wname, qty in rows]


# ---------------------------------------------------------------------------
# Generic Add/Edit form, driven by TABLES[...]["add_fields"]
# ---------------------------------------------------------------------------
class RecordForm(tk.Toplevel):
    def __init__(self, parent, table_name, mode, on_success, existing=None):
        super().__init__(parent)
        self.table_name = table_name
        self.meta = TABLES[table_name]
        self.mode = mode  # "add" or "edit"
        self.on_success = on_success
        self.existing = existing or {}
        self.widgets = {}

        verb = "Edit" if mode == "edit" else self.meta.get("add_button_label", "Add")
        self.title(f"{verb} -- {TAB_TITLES.get(table_name, table_name)}")
        self.resizable(False, False)
        self.grab_set()

        row = 0
        for field in self.meta["add_fields"]:
            col, label, kind = field["column"], field["label"], field["kind"]
            required = field.get("required", True)
            tk.Label(self, text=label + (":" if required else " (optional):")).grid(
                row=row, column=0, padx=8, pady=6, sticky="e")

            if kind == "fk":
                choices = run(field["choices_query"], fetch=True) or []
                if not choices:
                    messagebox.showwarning("No data", f"No rows found for '{label}' yet.")
                display_to_id = {f"{name} (ID {i})": i for i, name in choices}
                id_to_display = {i: f"{name} (ID {i})" for i, name in choices}
                var = tk.StringVar()
                if mode == "edit" and col in self.existing:
                    var.set(id_to_display.get(self.existing[col], ""))
                cb = ttk.Combobox(self, textvariable=var, values=list(display_to_id.keys()),
                                   state="readonly", width=32)
                cb.grid(row=row, column=1, padx=8, pady=6)
                self.widgets[col] = ("fk", (var, display_to_id), required)
            else:
                e = tk.Entry(self, width=34)
                if mode == "edit" and col in self.existing and self.existing[col] is not None:
                    e.insert(0, str(self.existing[col]))
                e.grid(row=row, column=1, padx=8, pady=6)
                self.widgets[col] = (kind, e, required)
            row += 1

        btn_frame = tk.Frame(self)
        btn_frame.grid(row=row, column=0, columnspan=2, pady=10)
        tk.Button(btn_frame, text="Save", width=12, command=self.save).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Cancel", width=12, command=self.destroy).pack(side="left", padx=5)

    def save(self):
        values = {}
        for col, (kind, widget, required) in self.widgets.items():
            try:
                if kind == "fk":
                    var, display_to_id = widget
                    choice = var.get()
                    if choice not in display_to_id:
                        if required:
                            raise ValueError(f"Please choose a '{col}'.")
                        continue
                    values[col] = display_to_id[choice]
                else:
                    raw = widget.get()
                    if raw.strip() == "" and not required:
                        continue
                    values[col] = VALIDATORS[kind](raw, col)
            except ValueError as ve:
                messagebox.showerror("Invalid input", str(ve))
                return

        try:
            if self.mode == "add":
                pk = self.meta["pk"]
                fixed = self.meta.get("fixed_values_on_insert", {})
                new_id = next_id(self.table_name, pk)
                all_values = {**values, **fixed}
                columns = [pk] + list(all_values.keys())
                placeholders = ", ".join(["%s"] * len(columns))
                params = [new_id] + list(all_values.values())
                run(f"INSERT INTO {self.table_name} ({', '.join(columns)}) VALUES ({placeholders})", params)
                messagebox.showinfo("Done", f"Added (ID {new_id}).")
            else:
                pk = self.meta["pk"]
                pk_value = self.existing[pk]
                set_clause = ", ".join(f"{c} = %s" for c in values.keys())
                params = list(values.values()) + [pk_value]
                run(f"UPDATE {self.table_name} SET {set_clause} WHERE {pk} = %s", params)
                messagebox.showinfo("Done", "Updated.")
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        self.on_success()
        self.destroy()


# ---------------------------------------------------------------------------
# Generic list tab: Search + Refresh + Add + Edit + Delete
# ---------------------------------------------------------------------------
class SearchableCrudTab(tk.Frame):
    allow_edit = True
    allow_add = True

    def __init__(self, parent, table_name):
        super().__init__(parent)
        self.table_name = table_name
        self.meta = TABLES[table_name]
        self.all_rows = []

        search_frame = tk.Frame(self)
        search_frame.pack(fill="x", padx=10, pady=(10, 0))
        tk.Label(search_frame, text="Search:").pack(side="left")
        self.search_var = tk.StringVar()
        search_entry = tk.Entry(search_frame, textvariable=self.search_var, width=30)
        search_entry.pack(side="left", padx=5)
        search_entry.bind("<Return>", lambda e: self.apply_search())
        tk.Button(search_frame, text="Search", command=self.apply_search).pack(side="left", padx=2)
        tk.Button(search_frame, text="Clear", command=self.refresh).pack(side="left", padx=2)

        tree_frame = tk.Frame(self)
        tree_frame.pack(fill="both", expand=True, padx=10, pady=10)

        self.tree = ttk.Treeview(tree_frame, columns=self.meta["labels"], show="headings", height=12)
        for label in self.meta["labels"]:
            self.tree.heading(label, text=label)
            self.tree.column(label, width=130, anchor="center")
        self.tree.pack(side="left", fill="both", expand=True)

        vsb = ttk.Scrollbar(tree_frame, orient="vertical", command=self.tree.yview)
        vsb.pack(side="right", fill="y")
        self.tree.configure(yscrollcommand=vsb.set)

        btn_frame = tk.Frame(self)
        btn_frame.pack(pady=6)
        tk.Button(btn_frame, text="Refresh / View All", width=16, command=self.refresh).pack(side="left", padx=4)
        if self.allow_add:
            tk.Button(btn_frame, text=self.meta.get("add_button_label", "Add"), width=16,
                      command=self.open_add_form).pack(side="left", padx=4)
        if self.allow_edit:
            tk.Button(btn_frame, text="Edit Selected", width=16, command=self.open_edit_form).pack(side="left", padx=4)
        tk.Button(btn_frame, text="Delete Selected", width=16, command=self.delete_selected).pack(side="left", padx=4)
        self.extra_buttons(btn_frame)

        self.refresh()

    def extra_buttons(self, btn_frame):
        """Hook for subclasses to add extra buttons."""
        pass

    def render_rows(self, rows):
        for row in self.tree.get_children():
            self.tree.delete(row)
        for row in rows:
            self.tree.insert("", "end", iid=str(row[0]), values=row)

    def refresh(self):
        self.search_var.set("")
        try:
            self.all_rows = run(self.meta["display_query"], fetch=True) or []
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return
        self.render_rows(self.all_rows)

    def apply_search(self):
        term = self.search_var.get().strip().lower()
        if term == "":
            self.render_rows(self.all_rows)
            return
        filtered = [r for r in self.all_rows if any(term in str(cell).lower() for cell in r)]
        self.render_rows(filtered)

    def selected_pk(self):
        sel = self.tree.selection()
        if not sel:
            messagebox.showwarning("No selection", "Select a row first.")
            return None
        return sel[0]

    def open_add_form(self):
        RecordForm(self, self.table_name, "add", on_success=self.refresh)

    def open_edit_form(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        rows = run_dict(f"SELECT * FROM {self.table_name} WHERE {self.meta['pk']} = %s", (pk_value,))
        if not rows:
            messagebox.showerror("Not found", "That record no longer exists.")
            return
        RecordForm(self, self.table_name, "edit", on_success=self.refresh, existing=rows[0])

    def delete_selected(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        if not messagebox.askyesno("Confirm delete", f"Delete this row (ID {pk_value})?"):
            return
        try:
            run(f"DELETE FROM {self.table_name} WHERE {self.meta['pk']} = %s", (pk_value,))
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return
        messagebox.showinfo("Done", "Deleted.")
        self.refresh()


class StockTab(SearchableCrudTab):
    """Adds Put-away (bin assignment) on top of the generic CRUD/Search."""

    def extra_buttons(self, btn_frame):
        tk.Button(btn_frame, text="Put-away (Set Bin)", width=16, command=self.put_away).pack(side="left", padx=4)

    def put_away(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        bin_value = simpledialog.askstring("Put-away", "Bin location (e.g. A1-03):")
        if bin_value is None:
            return
        try:
            run("UPDATE stock SET binLocation = %s WHERE stockID = %s", (bin_value.strip(), pk_value))
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return
        messagebox.showinfo("Done", "Bin location updated.")
        self.refresh()


class TransferTab(SearchableCrudTab):
    """Add = Initiate (status Pending). Adds a Confirm button that actually
    moves stock from the source to the destination warehouse."""

    def __init__(self, parent, table_name, on_stock_change):
        self.on_stock_change = on_stock_change
        super().__init__(parent, table_name)

    def extra_buttons(self, btn_frame):
        tk.Button(btn_frame, text="Confirm Selected", width=16, command=self.confirm_selected).pack(side="left", padx=4)

    def _status_of(self, transfer_id):
        rows = run("SELECT status FROM transfer WHERE transferID = %s", (transfer_id,), fetch=True)
        return rows[0][0] if rows else None

    def open_edit_form(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        if self._status_of(pk_value) == "Confirmed":
            messagebox.showwarning("Not allowed", "Confirmed transfers can't be edited.")
            return
        super().open_edit_form()

    def delete_selected(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        if self._status_of(pk_value) == "Confirmed":
            messagebox.showwarning("Not allowed", "Confirmed transfers can't be deleted (stock already moved).")
            return
        super().delete_selected()

    def confirm_selected(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        transfer_id = int(pk_value)
        if not messagebox.askyesno("Confirm transfer",
                                    f"Confirm transfer #{transfer_id}? This will move the actual stock "
                                    f"quantities between warehouses."):
            return

        def _do(cur):
            cur.execute("SELECT status, from_warehouseID, to_warehouseID FROM transfer "
                        "WHERE transferID = %s FOR UPDATE", (transfer_id,))
            row = cur.fetchone()
            if row is None:
                raise ValueError("Transfer not found.")
            status, from_wh, to_wh = row
            if status == "Confirmed":
                raise ValueError("This transfer is already confirmed.")

            cur.execute("SELECT productID, quantity FROM transferDetails WHERE transferID = %s", (transfer_id,))
            lines = cur.fetchall()
            if not lines:
                raise ValueError("This transfer has no line items yet -- add products in Transfer Details first.")

            for product_id, qty in lines:
                cur.execute("SELECT quantity FROM stock WHERE productID = %s AND warehouseID = %s FOR UPDATE",
                            (product_id, from_wh))
                src = cur.fetchone()
                available = src[0] if src else 0
                if available < qty:
                    raise ValueError(f"Not enough stock of product {product_id} in the source warehouse "
                                      f"(have {available}, need {qty}).")

            for product_id, qty in lines:
                cur.execute("UPDATE stock SET quantity = quantity - %s WHERE productID = %s AND warehouseID = %s",
                            (qty, product_id, from_wh))
                cur.execute("SELECT stockID FROM stock WHERE productID = %s AND warehouseID = %s",
                            (product_id, to_wh))
                dest = cur.fetchone()
                if dest:
                    cur.execute("UPDATE stock SET quantity = quantity + %s WHERE stockID = %s", (qty, dest[0]))
                else:
                    new_stock_id = next_id("stock", "stockID", cur)
                    cur.execute("INSERT INTO stock (stockID, warehouseID, productID, quantity) "
                                "VALUES (%s,%s,%s,%s)", (new_stock_id, to_wh, product_id, qty))

            cur.execute("UPDATE transfer SET status = 'Confirmed' WHERE transferID = %s", (transfer_id,))

        try:
            with_transaction(_do)
        except ValueError as ve:
            messagebox.showerror("Can't confirm transfer", str(ve))
            return
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        messagebox.showinfo("Done", f"Transfer #{transfer_id} confirmed and stock moved.")
        self.refresh()
        self.on_stock_change()


# ---------------------------------------------------------------------------
# Dispatch / Damage Entry / Reconciliation: audit-log tabs tied to a real
# stock movement. Add uses a dedicated form (not the generic RecordForm)
# because saving here also has to update the stock table.
# ---------------------------------------------------------------------------
class MovementForm(tk.Toplevel):
    """Base popup for Dispatch / Damage entries: pick a stock row, enter a
    quantity that can't exceed what's available, plus a couple of text fields."""

    def __init__(self, parent, title, extra_fields, on_success):
        super().__init__(parent)
        self.title(title)
        self.resizable(False, False)
        self.grab_set()
        self.on_success = on_success
        self.extra_fields = extra_fields  # list of (label, default_text)
        self.extra_widgets = {}

        self.choices = stock_choices()
        display_to_id = {label: sid for sid, label in self.choices}
        self.display_to_id = display_to_id

        tk.Label(self, text="Stock item:").grid(row=0, column=0, padx=8, pady=6, sticky="e")
        self.stock_var = tk.StringVar()
        cb = ttk.Combobox(self, textvariable=self.stock_var, values=list(display_to_id.keys()),
                           state="readonly", width=36)
        cb.grid(row=0, column=1, padx=8, pady=6)

        tk.Label(self, text="Quantity:").grid(row=1, column=0, padx=8, pady=6, sticky="e")
        self.qty_entry = tk.Entry(self, width=38)
        self.qty_entry.grid(row=1, column=1, padx=8, pady=6)

        row = 2
        for label, default in extra_fields:
            tk.Label(self, text=label + ":").grid(row=row, column=0, padx=8, pady=6, sticky="e")
            e = tk.Entry(self, width=38)
            if default:
                e.insert(0, default)
            e.grid(row=row, column=1, padx=8, pady=6)
            self.extra_widgets[label] = e
            row += 1

        btn_frame = tk.Frame(self)
        btn_frame.grid(row=row, column=0, columnspan=2, pady=10)
        tk.Button(btn_frame, text="Save", width=12, command=self.save).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Cancel", width=12, command=self.destroy).pack(side="left", padx=5)

    def get_common(self):
        """Validates and returns (stock_id, quantity, {label: value})."""
        choice = self.stock_var.get()
        if choice not in self.display_to_id:
            raise ValueError("Please choose a stock item.")
        stock_id = self.display_to_id[choice]
        qty = v_int(self.qty_entry.get(), "Quantity", minimum=1)
        extras = {}
        for label, widget in self.extra_widgets.items():
            extras[label] = v_required(widget.get(), label)
        return stock_id, qty, extras

    def save(self):
        raise NotImplementedError


class DispatchForm(MovementForm):
    def __init__(self, parent, on_success):
        super().__init__(parent, "Dispatch Stock",
                          [("Destination", ""), ("Date (YYYY-MM-DD)", date.today().isoformat())],
                          on_success)

    def save(self):
        try:
            stock_id, qty, extras = self.get_common()
            destination = extras["Destination"]
            dispatch_date = v_date(extras["Date (YYYY-MM-DD)"], "Date")
        except ValueError as ve:
            messagebox.showerror("Invalid input", str(ve))
            return

        def _do(cur):
            cur.execute("SELECT quantity FROM stock WHERE stockID = %s FOR UPDATE", (stock_id,))
            row = cur.fetchone()
            if row is None:
                raise ValueError("That stock item no longer exists.")
            available = row[0]
            if qty > available:
                raise ValueError(f"Only {available} available -- can't dispatch {qty}.")
            new_id = next_id("dispatch", "dispatchID", cur)
            cur.execute("UPDATE stock SET quantity = quantity - %s WHERE stockID = %s", (qty, stock_id))
            cur.execute("INSERT INTO dispatch (dispatchID, stockID, quantity, destination, dispatchDate) "
                        "VALUES (%s,%s,%s,%s,%s)", (new_id, stock_id, qty, destination, dispatch_date))
            return new_id

        try:
            new_id = with_transaction(_do)
        except ValueError as ve:
            messagebox.showerror("Can't dispatch", str(ve))
            return
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        messagebox.showinfo("Done", f"Dispatched (ID {new_id}). Stock updated.")
        self.on_success()
        self.destroy()


class DamageForm(MovementForm):
    def __init__(self, parent, on_success):
        super().__init__(parent, "Damage Entry",
                          [("Reason", ""), ("Date (YYYY-MM-DD)", date.today().isoformat())],
                          on_success)

    def save(self):
        try:
            stock_id, qty, extras = self.get_common()
            reason = extras["Reason"]
            damage_date = v_date(extras["Date (YYYY-MM-DD)"], "Date")
        except ValueError as ve:
            messagebox.showerror("Invalid input", str(ve))
            return

        def _do(cur):
            cur.execute("SELECT quantity FROM stock WHERE stockID = %s FOR UPDATE", (stock_id,))
            row = cur.fetchone()
            if row is None:
                raise ValueError("That stock item no longer exists.")
            available = row[0]
            if qty > available:
                raise ValueError(f"Only {available} available -- can't mark {qty} as damaged.")
            new_id = next_id("damagedStock", "damageID", cur)
            cur.execute("UPDATE stock SET quantity = quantity - %s WHERE stockID = %s", (qty, stock_id))
            cur.execute("INSERT INTO damagedStock (damageID, stockID, quantity, reason, damageDate) "
                        "VALUES (%s,%s,%s,%s,%s)", (new_id, stock_id, qty, reason, damage_date))
            return new_id

        try:
            new_id = with_transaction(_do)
        except ValueError as ve:
            messagebox.showerror("Can't record damage", str(ve))
            return
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        messagebox.showinfo("Done", f"Damage recorded (ID {new_id}). Stock updated.")
        self.on_success()
        self.destroy()


class ReconciliationForm(tk.Toplevel):
    def __init__(self, parent, on_success):
        super().__init__(parent)
        self.title("Reconciliation")
        self.resizable(False, False)
        self.grab_set()
        self.on_success = on_success

        self.choices = stock_choices()
        self.display_to_id = {label: sid for sid, label in self.choices}

        tk.Label(self, text="Stock item:").grid(row=0, column=0, padx=8, pady=6, sticky="e")
        self.stock_var = tk.StringVar()
        cb = ttk.Combobox(self, textvariable=self.stock_var, values=list(self.display_to_id.keys()),
                           state="readonly", width=36)
        cb.grid(row=0, column=1, padx=8, pady=6)

        tk.Label(self, text="Counted Quantity:").grid(row=1, column=0, padx=8, pady=6, sticky="e")
        self.counted_entry = tk.Entry(self, width=38)
        self.counted_entry.grid(row=1, column=1, padx=8, pady=6)

        tk.Label(self, text="Note (optional):").grid(row=2, column=0, padx=8, pady=6, sticky="e")
        self.note_entry = tk.Entry(self, width=38)
        self.note_entry.grid(row=2, column=1, padx=8, pady=6)

        self.apply_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self, text="Update system quantity to match the count",
                        variable=self.apply_var).grid(row=3, column=0, columnspan=2, pady=4)

        btn_frame = tk.Frame(self)
        btn_frame.grid(row=4, column=0, columnspan=2, pady=10)
        tk.Button(btn_frame, text="Save", width=12, command=self.save).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Cancel", width=12, command=self.destroy).pack(side="left", padx=5)

    def save(self):
        choice = self.stock_var.get()
        if choice not in self.display_to_id:
            messagebox.showerror("Invalid input", "Please choose a stock item.")
            return
        stock_id = self.display_to_id[choice]
        try:
            counted_qty = v_int(self.counted_entry.get(), "Counted Quantity", minimum=0)
        except ValueError as ve:
            messagebox.showerror("Invalid input", str(ve))
            return
        note = self.note_entry.get().strip()
        apply_adjustment = self.apply_var.get()
        recon_date = date.today().isoformat()

        def _do(cur):
            cur.execute("SELECT quantity FROM stock WHERE stockID = %s FOR UPDATE", (stock_id,))
            row = cur.fetchone()
            if row is None:
                raise ValueError("That stock item no longer exists.")
            system_qty = row[0]
            difference = counted_qty - system_qty
            new_id = next_id("reconciliation", "reconID", cur)
            cur.execute("INSERT INTO reconciliation "
                        "(reconID, stockID, systemQty, countedQty, difference, reconDate, note) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s)",
                        (new_id, stock_id, system_qty, counted_qty, difference, recon_date, note))
            if apply_adjustment:
                cur.execute("UPDATE stock SET quantity = %s WHERE stockID = %s", (counted_qty, stock_id))
            return new_id, difference

        try:
            new_id, difference = with_transaction(_do)
        except ValueError as ve:
            messagebox.showerror("Can't reconcile", str(ve))
            return
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        messagebox.showinfo("Done", f"Reconciliation recorded (ID {new_id}). Difference: {difference:+d}.")
        self.on_success()
        self.destroy()


class MovementTab(SearchableCrudTab):
    """Shared behaviour for Dispatch / Damage / Reconciliation tabs:
    custom Add form, no Edit, Delete reverses the stock effect."""
    allow_edit = False
    table_name = None
    pk_col = None
    form_class = None
    reverses_stock = True  # Reconciliation overrides this to False

    def __init__(self, parent, on_stock_change):
        self.on_stock_change = on_stock_change
        super().__init__(parent, self.table_name)

    def open_add_form(self):
        self.form_class(self, on_success=self._after_change)

    def _after_change(self):
        self.refresh()
        self.on_stock_change()

    def delete_selected(self):
        pk_value = self.selected_pk()
        if pk_value is None:
            return
        note = " This will ADD the quantity back to stock." if self.reverses_stock else ""
        if not messagebox.askyesno("Confirm delete", f"Delete this record (ID {pk_value})?{note}"):
            return

        if not self.reverses_stock:
            try:
                run(f"DELETE FROM {self.table_name} WHERE {self.pk_col} = %s", (pk_value,))
            except mysql.connector.Error as e:
                messagebox.showerror("Database error", friendly_db_error(e))
                return
            messagebox.showinfo("Done", "Deleted.")
            self.refresh()
            return

        def _do(cur):
            cur.execute(f"SELECT stockID, quantity FROM {self.table_name} WHERE {self.pk_col} = %s", (pk_value,))
            row = cur.fetchone()
            if row is None:
                raise ValueError("Record not found.")
            stock_id, qty = row
            cur.execute("UPDATE stock SET quantity = quantity + %s WHERE stockID = %s", (qty, stock_id))
            cur.execute(f"DELETE FROM {self.table_name} WHERE {self.pk_col} = %s", (pk_value,))

        try:
            with_transaction(_do)
        except ValueError as ve:
            messagebox.showerror("Can't delete", str(ve))
            return
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return

        messagebox.showinfo("Done", "Deleted and stock restored.")
        self._after_change()


class DispatchTab(MovementTab):
    table_name = "dispatch"
    pk_col = "dispatchID"
    form_class = DispatchForm


class DamageTab(MovementTab):
    table_name = "damagedStock"
    pk_col = "damageID"
    form_class = DamageForm


class ReconciliationTab(MovementTab):
    table_name = "reconciliation"
    pk_col = "reconID"
    form_class = ReconciliationForm
    reverses_stock = False


TABLES["dispatch"] = {
    "display_query": """
        SELECT d.dispatchID, w.warehouseName, p.productName, d.quantity, d.destination, d.dispatchDate
        FROM dispatch d
        JOIN stock s ON d.stockID = s.stockID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        JOIN product p ON s.productID = p.productID
    """,
    "labels": ["ID", "Warehouse", "Product", "Quantity", "Destination", "Date"],
}
TABLES["damagedStock"] = {
    "display_query": """
        SELECT d.damageID, w.warehouseName, p.productName, d.quantity, d.reason, d.damageDate
        FROM damagedStock d
        JOIN stock s ON d.stockID = s.stockID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        JOIN product p ON s.productID = p.productID
    """,
    "labels": ["ID", "Warehouse", "Product", "Quantity", "Reason", "Date"],
}
TABLES["reconciliation"] = {
    "display_query": """
        SELECT r.reconID, w.warehouseName, p.productName, r.systemQty, r.countedQty,
               r.difference, r.reconDate, r.note
        FROM reconciliation r
        JOIN stock s ON r.stockID = s.stockID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        JOIN product p ON s.productID = p.productID
    """,
    "labels": ["ID", "Warehouse", "Product", "System Qty", "Counted Qty", "Difference", "Date", "Note"],
}


# ---------------------------------------------------------------------------
# Report tab: several report types in one place
# ---------------------------------------------------------------------------
REPORTS = {
    "Stock by Warehouse": """
        SELECT w.warehouseName, SUM(s.quantity) AS total
        FROM stock s JOIN warehouse w ON s.warehouseID = w.warehouseID
        GROUP BY w.warehouseName
    """,
    "Stock by Product": """
        SELECT p.productName, SUM(s.quantity) AS total
        FROM stock s JOIN product p ON s.productID = p.productID
        GROUP BY p.productName
    """,
    "Low Stock Alert (< 10)": """
        SELECT p.productName, w.warehouseName, s.quantity
        FROM stock s
        JOIN product p ON s.productID = p.productID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        WHERE s.quantity < 10
        ORDER BY s.quantity
    """,
    "Damaged Goods Summary": """
        SELECT p.productName, SUM(d.quantity) AS total_damaged
        FROM damagedStock d
        JOIN stock s ON d.stockID = s.stockID
        JOIN product p ON s.productID = p.productID
        GROUP BY p.productName
    """,
    "Reconciliation Discrepancies": """
        SELECT p.productName, w.warehouseName, r.systemQty, r.countedQty, r.difference, r.reconDate
        FROM reconciliation r
        JOIN stock s ON r.stockID = s.stockID
        JOIN product p ON s.productID = p.productID
        JOIN warehouse w ON s.warehouseID = w.warehouseID
        WHERE r.difference <> 0
        ORDER BY r.reconDate DESC
    """,
    "Pending Transfers": """
        SELECT t.transferID, wf.warehouseName, wt.warehouseName, t.transferData
        FROM transfer t
        JOIN warehouse wf ON t.from_warehouseID = wf.warehouseID
        JOIN warehouse wt ON t.to_warehouseID = wt.warehouseID
        WHERE t.status = 'Pending'
    """,
}


class ReportTab(tk.Frame):
    def __init__(self, parent):
        super().__init__(parent)
        top = tk.Frame(self)
        top.pack(fill="x", padx=10, pady=10)
        tk.Label(top, text="Report:").pack(side="left")
        self.report_var = tk.StringVar(value=list(REPORTS.keys())[0])
        cb = ttk.Combobox(top, textvariable=self.report_var, values=list(REPORTS.keys()),
                           state="readonly", width=32)
        cb.pack(side="left", padx=6)
        tk.Button(top, text="Run Report", command=self.run_report).pack(side="left", padx=6)

        self.text = tk.Text(self, width=90, height=22)
        self.text.pack(padx=10, pady=(0, 10), fill="both", expand=True)
        self.run_report()

    def run_report(self):
        name = self.report_var.get()
        sql = REPORTS[name]
        self.text.delete("1.0", "end")
        try:
            rows = run(sql, fetch=True) or []
        except mysql.connector.Error as e:
            messagebox.showerror("Database error", friendly_db_error(e))
            return
        self.text.insert("end", f"=== {name} ===\n\n")
        if not rows:
            self.text.insert("end", "(no rows)\n")
            return
        for row in rows:
            self.text.insert("end", " | ".join(str(v) for v in row) + "\n")


# ---------------------------------------------------------------------------
# Main window
# ---------------------------------------------------------------------------
def main():
    root = tk.Tk()
    root.title("Warehouse Management System")
    root.geometry("900x620")

    tk.Label(root, text="Warehouse Management System", font=("Arial", 18, "bold")).pack(pady=10)

    notebook = ttk.Notebook(root)
    notebook.pack(fill="both", expand=True, padx=10, pady=10)

    for name in ["warehouse", "supplier", "product"]:
        notebook.add(SearchableCrudTab(notebook, name), text=TAB_TITLES[name])

    stock_tab = StockTab(notebook, "stock")
    notebook.add(stock_tab, text="Stock")

    notebook.add(TransferTab(notebook, "transfer", on_stock_change=stock_tab.refresh), text="Transfers")
    notebook.add(SearchableCrudTab(notebook, "transferDetails"), text="Transfer Details")

    notebook.add(DispatchTab(notebook, on_stock_change=stock_tab.refresh), text="Dispatch")
    notebook.add(DamageTab(notebook, on_stock_change=stock_tab.refresh), text="Damage Entry")
    notebook.add(ReconciliationTab(notebook, on_stock_change=stock_tab.refresh), text="Reconciliation")

    notebook.add(ReportTab(notebook), text="Report")

    root.mainloop()


if __name__ == "__main__":
    main()